# PSM-DID Analysis of the Addis Ababa-Djibouti Railway

**Purpose.** Estimate the railway-opening effect using propensity-score matching, two-way fixed effects, and event-study checks.

**Inputs.** Transport, macroeconomic, sectoral, and administrative-boundary data defined in the notebook.

**Outputs.** Matching diagnostics, DID estimates, event-study estimates, and figures.

Run the code cells from top to bottom after updating the local paths described in the repository README. The notebook has no stored outputs; figures and tables are generated when it is executed.

In [ ]:
# Cell | PSM-DID design and analysis dependencies

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['figure.dpi'] = 150

from linearmodels.panel import PanelOLS
from scipy.special import expit                  # Logistic sigmoid
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

print("=" * 60)
print("=" * 60)


In [ ]:
# Cell | Transport, economic, sectoral and spatial data assembly

region_mapping = {
    'North Africa':    ['Algeria', 'Egypt', 'Libya', 'Mauritania', 'Morocco', 'Tunisia'],
    'West Africa':     ['Benin', 'Burkina Faso', 'Ivory Coast', 'Gambia', 'Ghana',
                        'Guinea', 'Guinea-Bissau', 'Liberia', 'Mali', 'Niger',
                        'Nigeria', 'Senegal', 'Sierra Leone', 'Togo'],
    'Central Africa':  ['Cameroon', 'Central African Republic', 'Chad', 'Republic of the Congo',
                        'Democratic Republic of the Congo', 'Equatorial Guinea', 'Gabon'],
    'East Africa':     ['Burundi', 'Djibouti', 'Eritrea', 'Ethiopia', 'Kenya',
                        'Rwanda', 'Somalia', 'South Sudan', 'Sudan',
                        'United Republic of Tanzania', 'Uganda'],
    'Southern Africa': ['Angola', 'Botswana', 'eSwatini', 'Lesotho', 'Madagascar', 'Malawi',
                        'Mozambique', 'Namibia', 'South Africa', 'Zambia', 'Zimbabwe'],
}
country_to_region    = {c: r for r, cs in region_mapping.items() for c in cs}
all_mapped_countries = list(country_to_region.keys())

df_transport = pd.read_excel(
    r'E:\Africa_Infrastructure_Data\transport_statistics_2015_2025.xlsx'
)
df_transport = df_transport[
    (df_transport['year'] >= 2015) & (df_transport['year'] <= 2024)
].copy()
df_transport['region'] = df_transport['country'].map(country_to_region)
df_transport = df_transport[df_transport['country'].isin(all_mapped_countries)].copy()

road_type_cols = ['road_motorway_km', 'road_trunk_km', 'road_primary_km',
                  'road_secondary_km', 'road_tertiary_km']
df_transport['total_road_km'] = df_transport[road_type_cols].sum(axis=1)

df_econ_raw = pd.read_csv(r'E:\Africa_Infrastructure_Data\afdb\ObservationData_fhkvkjf_constant.csv')

afdb_map = {
    "Congo, Dem. Rep.": "Democratic Republic of the Congo",
    "Congo, Rep.":      "Republic of the Congo",
    "Tanzania":         "United Republic of Tanzania",
    "Eswatini":         "eSwatini",
    "Cote d'Ivoire":    "Ivory Coast",
}
df_econ_raw['country'] = df_econ_raw['country'].replace(afdb_map)
df_econ_raw = df_econ_raw[
    (df_econ_raw['Date'] >= 2015) & (df_econ_raw['Date'] <= 2024)
]

def get_indicator(indicator_name, value_col='Value'):
    """Extract an indicator from AfDB data."""
    df = df_econ_raw[df_econ_raw['indicator'] == indicator_name][
        ['country', 'Date', value_col]
    ].copy()
    return df.rename(columns={'Date': 'year', value_col: indicator_name})

df_gdp     = get_indicator('GDP (constant 2000 US$)')
df_gdppc   = get_indicator('GDP per capita,(constant 2000 Prices, US $)')
df_pop_raw = get_indicator('Population, total')

_pop_df  = df_econ_raw[df_econ_raw['indicator'] == 'Population, total'][
    ['country', 'Date', 'Value']].rename(columns={'Date': 'year', 'Value': 'Pop'})
_hfce_df = df_econ_raw[df_econ_raw['indicator'] ==
    'Household final consumption expenditure (current US$)'][
    ['country', 'Date', 'Value']].rename(columns={'Date': 'year', 'Value': 'HFCE'})
_er_df   = df_econ_raw[df_econ_raw['indicator'] ==
    'Exchange rate (LCU per US$, period average)'][
    ['country', 'Date', 'Value']].rename(columns={'Date': 'year', 'Value': 'ER'})
_cpi_df  = df_econ_raw[df_econ_raw['indicator'] ==
    'Inflation, consumer prices  index (2000 = 100)'][
    ['country', 'Date', 'Value']].rename(columns={'Date': 'year', 'Value': 'CPI'})

_mc = (_pop_df
       .merge(_hfce_df, on=['country', 'year'], how='inner')
       .merge(_er_df,   on=['country', 'year'], how='left')
       .merge(_cpi_df,  on=['country', 'year'], how='left'))
_mc['HFCE_pc_curr'] = _mc['HFCE'] / _mc['Pop']
_base = _mc[_mc['year'] == 2015][['country', 'ER', 'CPI']].rename(
    columns={'ER': 'ER_2015', 'CPI': 'CPI_2015'})
_mc = _mc.merge(_base, on='country', how='left')
_mc['ER_ratio']  = (_mc['ER']       / _mc['ER_2015'] ).fillna(1)
_mc['CPI_ratio'] = (_mc['CPI_2015'] / _mc['CPI']     ).fillna(1)
_mc['HFCE_PC']   = _mc['HFCE_pc_curr'] * _mc['ER_ratio'] * _mc['CPI_ratio']
df_hfce_pc = _mc[['country', 'year', 'HFCE_PC']]

industry_files = {
    'Agriculture': r'E:\Africa_Infrastructure_Data\world_bank\API_NV.AGR.TOTL.CD_DS2_en_csv_v2_723\API_NV.AGR.TOTL.CD_DS2_en_csv_v2_723.csv',
    'Industry':    r'E:\Africa_Infrastructure_Data\world_bank\API_NV.IND.TOTL.KD_DS2_en_csv_v2_14108\API_NV.IND.TOTL.KD_DS2_en_csv_v2_14108.csv',
    'Services':    r'E:\Africa_Infrastructure_Data\world_bank\API_NV.SRV.TOTL.KD_DS2_en_csv_v2_14118\API_NV.SRV.TOTL.KD_DS2_en_csv_v2_14118.csv',
}

wb_to_project = {
    'Congo, Dem. Rep.':  'Democratic Republic of the Congo',
    'Congo, Rep.':       'Republic of the Congo',
    'Tanzania':          'United Republic of Tanzania',
    'Eswatini':          'eSwatini',
    "Cote d'Ivoire":     'Ivory Coast',
    'Egypt, Arab Rep.':  'Egypt',
    'Gambia, The':       'Gambia',
}

year_cols_str = [str(y) for y in range(2015, 2025)]  # Define the variables used in this analysis block.

industry_data_list = []

for sector, fpath in industry_files.items():
    df_s = pd.read_csv(fpath, skiprows=4)
    df_s['Country Name'] = df_s['Country Name'].replace(wb_to_project)
    df_s = df_s[df_s['Country Name'].isin(all_mapped_countries)][
        ['Country Name', 'Country Code', 'Indicator Name'] + year_cols_str
    ]
    df_long = df_s.melt(
        id_vars=['Country Name', 'Country Code', 'Indicator Name'],
        value_vars=year_cols_str,
        var_name='year', value_name='value'
    )
    df_long['year'] = df_long['year'].astype(int)
    df_long['sector'] = sector

    valid = [
        c for c in df_s['Country Name'].unique()
        if df_long[(df_long['Country Name'] == c) & df_long['value'].notna()].shape[0] > 3
    ]
    industry_data_list.append(df_long[df_long['Country Name'].isin(valid)])

# Stack the equation-specific observations for joint estimation.
df_industry_long = pd.concat(industry_data_list, ignore_index=True)

# Reshape the long table into indicator columns.
df_industry_pivot = df_industry_long.pivot_table(
    index=['Country Name', 'year'], columns='sector', values='value', aggfunc='first'
).reset_index()
df_industry_pivot.columns.name = None

df_industry_pivot = df_industry_pivot.rename(columns={
    'Country Name': 'country',
    'Agriculture':  'Agri_va',
    'Industry':     'Indus_va',
    'Services':     'Serv_va',
})

df_industry_pivot['sector_sum'] = (
    df_industry_pivot['Agri_va'].fillna(0) +
    df_industry_pivot['Indus_va'].fillna(0) +
    df_industry_pivot['Serv_va'].fillna(0)
)
df_industry_pivot['Indus_pct'] = (
    df_industry_pivot['Indus_va'] / df_industry_pivot['sector_sum'] * 100
)
df_industry_pivot['Serv_pct'] = (
    df_industry_pivot['Serv_va'] / df_industry_pivot['sector_sum'] * 100
)
df_industry_pivot = df_industry_pivot.drop(columns=['sector_sum'])


import geopandas as gpd
gdf_shp = gpd.read_file(
    r'E:\Africa_Infrastructure_Data\boundaries_project_49\Country.shp'
)
gdf_shp['Land_Area_km2'] = gdf_shp.geometry.area / 1e6
shp_map = {"Côte d'Ivoire": 'Ivory Coast'}
gdf_shp['SOVEREIGNT'] = gdf_shp['SOVEREIGNT'].replace(shp_map)
df_area = gdf_shp[['SOVEREIGNT', 'Land_Area_km2']].rename(columns={'SOVEREIGNT': 'country'})

df_main = (df_transport
    .merge(df_gdp.rename(columns={'GDP (constant 2000 US$)': 'GDP'}),
           on=['country', 'year'], how='left')
    .merge(df_gdppc.rename(columns={'GDP per capita,(constant 2000 Prices, US $)': 'GDP_PC'}),
           on=['country', 'year'], how='left')
    .merge(df_hfce_pc,       on=['country', 'year'], how='left')
    .merge(df_industry_pivot, on=['country', 'year'], how='left')  # Merge records using the shared country-year keys.
    .merge(df_pop_raw.rename(columns={'Population, total': 'Population'}),
           on=['country', 'year'], how='left')
    .merge(df_area, on='country', how='left')
)
df_main = df_main[df_main['country'].isin(all_mapped_countries)].copy()

print(f"          Indus_pct, Serv_pct, total_road_km, motorway_trunk_paved_km")
print(f"          railway_length_km, Population, Land_Area_km2")


In [ ]:
# Cell | Construction of an uncontaminated control pool

# ============================================================================
# Define treatment timing, treated countries and the railway-growth exclusion threshold.
RAILWAY_DELTA_THRESHOLD = 100
POLICY_YEAR             = 2018
TREAT_COUNTRIES         = ['Ethiopia', 'Djibouti']

HARD_EXCLUDE = ['Nigeria', 'Kenya', 'Morocco', 'Senegal', 'United Republic of Tanzania']

all_countries = df_main['country'].unique().tolist()
candidate_pool = [
    c for c in all_countries
    if c not in TREAT_COUNTRIES and c not in HARD_EXCLUDE
]



df_rail = df_main[df_main['country'].isin(candidate_pool)][
    ['country', 'year', 'railway_length_km']
].copy()

rail_2015 = (df_rail[df_rail['year'] == df_rail['year'].min()]
             .groupby('country')['railway_length_km'].mean())
rail_2024 = (df_rail[df_rail['year'] == df_rail['year'].max()]
             .groupby('country')['railway_length_km'].mean())

rail_delta = (rail_2024 - rail_2015).rename('rail_delta').reset_index()
rail_delta['rail_delta_abs'] = rail_delta['rail_delta'].abs()

print(rail_delta.sort_values('rail_delta', ascending=False)
      .to_string(index=False))

excluded_by_delta = rail_delta[
    rail_delta['rail_delta_abs'] > RAILWAY_DELTA_THRESHOLD
]['country'].tolist()

pure_control_pool = [
    c for c in candidate_pool
    if c not in excluded_by_delta
]

print(f"  {pure_control_pool}")


In [ ]:
# Cell | Pre-treatment propensity-score matching

# ============================================================================
#   ln(GDP), ln(Population), ln(motorway_trunk_paved_km), ln(Land_Area_km2)
# ============================================================================

# Use 2015–2017 covariates to define the pre-opening matching baseline.
PRE_YEARS = list(range(2015, POLICY_YEAR))   # [2015, 2016, 2017]

LOG_FEATURES_RAW = [
    'GDP',
    'Population',
    'Land_Area_km2',
]
RATIO_FEATURES = []
RAW_EXTRACT_COLS = LOG_FEATURES_RAW + RATIO_FEATURES

psm_countries = TREAT_COUNTRIES + pure_control_pool

df_pre = df_main[
    df_main['country'].isin(psm_countries) &
    df_main['year'].isin(PRE_YEARS)
].copy()

df_baseline = (df_pre
               .groupby('country')[RAW_EXTRACT_COLS]
               .mean()
               .reset_index())

for feat in LOG_FEATURES_RAW:
    df_baseline['ln_' + feat] = np.log(df_baseline[feat].clip(lower=1e-6))

df_baseline['treated'] = df_baseline['country'].isin(TREAT_COUNTRIES).astype(int)

LOG_PSM_COLS   = ['ln_' + f for f in LOG_FEATURES_RAW]
PSM_FEATURES   = LOG_PSM_COLS + RATIO_FEATURES

# Run model diagnostics.

cols_to_drop = []

for feat in PSM_FEATURES:
    total      = len(df_baseline)
    n_missing  = df_baseline[feat].isnull().sum()
    pct        = n_missing / total * 100
    status     = "✓ No missing values" if n_missing == 0 else (
                 "⚠ Partial missing values" if n_missing < total else "✗ All missing")
    print(f"    {feat:<35s}: {n_missing:2d}/{total} ({pct:.0f}%)  {status}")

    if n_missing == total:
        cols_to_drop.append(feat)
    elif n_missing > 0:
        for grp in [0, 1]:
            mask_grp = df_baseline['treated'] == grp
            grp_med  = df_baseline.loc[mask_grp, feat].median()
            fill_idx = mask_grp & df_baseline[feat].isnull()
            if fill_idx.sum() > 0:
                fill_val = grp_med if pd.notna(grp_med) else df_baseline[feat].median()
                df_baseline.loc[fill_idx, feat] = fill_val

if cols_to_drop:
    for c in cols_to_drop:
        print(f"    - {c}")
    PSM_FEATURES = [f for f in PSM_FEATURES if f not in cols_to_drop]
else:
    pass

# Retain complete observations required for estimation.
df_baseline_clean = df_baseline.dropna(subset=PSM_FEATURES).copy()

if len(df_baseline_clean) == 0:
    raise RuntimeError("PSM sample is empty! Please check the covariate coverage in df_main.")

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler   = StandardScaler()
X_raw    = df_baseline_clean[PSM_FEATURES].values
X_scaled = scaler.fit_transform(X_raw)
y_treat  = df_baseline_clean['treated'].values

logit_model = LogisticRegression(
    penalty=None,
    solver='lbfgs',
    max_iter=2000,
    random_state=42
)
logit_model.fit(X_scaled, y_treat)

df_baseline_clean = df_baseline_clean.copy()
df_baseline_clean['ps_score'] = logit_model.predict_proba(X_scaled)[:, 1]

ps_display = df_baseline_clean[['country', 'treated', 'ps_score']].sort_values(
    'ps_score', ascending=False)
for _, row in ps_display.iterrows():
    tag = "[Treated Group]" if row['treated'] == 1 else "  Control Group  "
    print(f"    {tag} {row['country']:<45} PS = {row['ps_score']:.4f}")

df_treat_ps   = df_baseline_clean[df_baseline_clean['treated'] == 1][
    ['country', 'ps_score']].reset_index(drop=True)
df_control_ps = df_baseline_clean[df_baseline_clean['treated'] == 0][
    ['country', 'ps_score']].reset_index(drop=True)

N_MATCH = 4

matched_control_list = []

for _, treat_row in df_treat_ps.iterrows():
    df_control_ps = df_control_ps.copy()
    df_control_ps['ps_dist'] = (
        df_control_ps['ps_score'] - treat_row['ps_score']
    ).abs()
    nearest = df_control_ps.nsmallest(N_MATCH, 'ps_dist')
    matched_control_list.extend(nearest['country'].tolist())

    for _, nr in nearest.iterrows():
        pass

final_control_countries = list(set(matched_control_list))

In [ ]:
# Cell | Covariate-balance diagnostics and Love plot
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 14
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['figure.dpi'] = 300


def compute_std_bias(df_treat_group, df_ctrl_group, features):
    """Helper function."""
    rows = []
    for feat in features:
        mu_t  = df_treat_group[feat].mean()
        mu_c  = df_ctrl_group[feat].mean()
        var_t = df_treat_group[feat].var()
        var_c = df_ctrl_group[feat].var()
        denom = np.sqrt((var_t + var_c) / 2)
        sb    = (mu_t - mu_c) / denom * 100 if denom > 1e-10 else 0.0
        rows.append({
            'Covariate':        feat,
            'Treated Mean':     round(mu_t, 4),
            'Control Mean':     round(mu_c, 4),
            'Standardized Bias (%)': round(sb,   2)
        })
    return pd.DataFrame(rows)

df_treat_bl       = df_baseline_clean[df_baseline_clean['treated'] == 1]
df_control_before = df_baseline_clean[df_baseline_clean['treated'] == 0]

df_control_after  = df_baseline_clean[
    df_baseline_clean['country'].isin(final_control_countries) &
    (df_baseline_clean['treated'] == 0)
]

sb_before = compute_std_bias(df_treat_bl, df_control_before, PSM_FEATURES)
sb_after  = compute_std_bias(df_treat_bl, df_control_after,  PSM_FEATURES)

sb_compare = sb_before[['Covariate', 'Treated Mean', 'Control Mean', 'Standardized Bias (%)']].copy()
sb_compare = sb_compare.rename(columns={
    'Control Mean':        'Control Mean Before Matching',
    'Standardized Bias (%)': 'SB (%) Before Matching'
})
sb_compare['Control Mean After Matching'] = sb_after['Control Mean'].values
sb_compare['SB (%) After Matching']       = sb_after['Standardized Bias (%)'].values
sb_compare['Balanced After Matching?'] = sb_compare['SB (%) After Matching'].abs().apply(
    lambda x: '✓ Passed' if x < 10 else '✗ Failed'
)

print("=" * 80)
print("=" * 80)
print(sb_compare.to_string(index=False))

# ── Love Plot ────────────────────────────────────────────────────────────────
labels_en_full = {
    'ln_GDP':                     'ln(GDP)',
    'ln_Population':              'ln(Population)',
    'ln_motorway_trunk_paved_km': 'ln(Paved Road)',
    'ln_Land_Area_km2':           'ln(Land Area)',
    'Indus_pct':                  'Industry VA (%)',
    'Serv_pct':                   'Services VA (%)',
    'ln_GDP_PC':                  'ln(GDP per capita)',
    'ln_total_road_km':           'ln(Total Road)',
    'ln_railway_length_km':       'ln(Railway)',
}
feat_labels = [labels_en_full.get(f, f) for f in PSM_FEATURES]

fig, ax = plt.subplots(figsize=(8, max(4, len(PSM_FEATURES) * 0.75)))
y_pos = np.arange(len(PSM_FEATURES))

ax.scatter(sb_compare['SB (%) Before Matching'], y_pos,
           color='#E07B39', s=110, zorder=5, label='Before matching',
           marker='o', facecolors='none', linewidths=2.2)
ax.scatter(sb_compare['SB (%) After Matching'], y_pos,
           color='#3A79B8', s=110, zorder=5, label='After matching', marker='o')

for i in range(len(PSM_FEATURES)):
    ax.plot([sb_compare['SB (%) Before Matching'].iloc[i],
             sb_compare['SB (%) After Matching'].iloc[i]],
            [i, i], color='#BBBBBB', lw=1.2, zorder=3)

ax.axvline(x= 10, color='#CC3333', lw=1.3, ls='--', alpha=0.75, label='±10% threshold')
ax.axvline(x=-10, color='#CC3333', lw=1.3, ls='--', alpha=0.75)
ax.axvline(x=  0, color='#444444', lw=0.9, ls='-',  alpha=0.45)

ax.set_yticks(y_pos)
ax.set_yticklabels(feat_labels, fontsize=14)
ax.set_xlabel('Standardized Bias (%)', fontsize=14)
# ax.set_title(f'Covariate Balance — Love Plot (PSM)\n'
#              f'Covariates ({len(PSM_FEATURES)}): {", ".join(feat_labels)}',
#              fontsize=11, fontweight='bold')
ax.legend(fontsize=12, loc='lower left')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.28, ls='--')
ax.set_xlim(-11, 15)
plt.tight_layout()
plt.show()

In [ ]:
# Cell | Matched DID panel and treatment indicators

# Combine treated countries with matched controls before constructing DID indicators.
final_did_countries = TREAT_COUNTRIES + final_control_countries

df_did = df_main[df_main['country'].isin(final_did_countries)].copy()

df_did['treated_dummy'] = df_did['country'].isin(TREAT_COUNTRIES).astype(int)

df_did['post_dummy']    = (df_did['year'] >= POLICY_YEAR).astype(int)

df_did['did_treat']     = df_did['treated_dummy'] * df_did['post_dummy']



In [ ]:
# Cell | Two-way fixed-effects DID estimation

outcome_vars = {
    'ln_GDP':      ('GDP',      'GDP (constant 2000 US dollars)'),
    'ln_GDP_PC':   ('GDP_PC',   'Per capita GDP (constant 2000 US dollars)'),
    'ln_HFCE_PC':  ('HFCE_PC',  'Per capita consumption (2015 constant US dollars)'),
    'ln_Agri_va':  ('Agri_va',  'Agricultural value added (World Bank current US dollars)'),
    'ln_Indus_va': ('Indus_va', 'Industrial value added (World Bank constant prices)'),
    'ln_Serv_va':  ('Serv_va',  'Service sector value added (World Bank constant prices)'),
}

for log_col, (raw_col, label) in outcome_vars.items():
    df_did[log_col] = np.log(df_did[raw_col].clip(lower=1e-6))

# Apply the specified logarithmic transformation.
df_did['ln_Population'] = np.log(df_did['Population'].clip(lower=1))
df_did['ln_Land_Area']  = np.log(df_did['Land_Area_km2'].clip(lower=1))
df_did['ln_road_km']    = np.log(df_did['total_road_km'].clip(lower=1))
df_did['ln_paved_km']   = np.log(df_did['motorway_trunk_paved_km'].clip(lower=1))

# Work on a copy to preserve the source table.
df_panel = df_did.set_index(['country', 'year']).copy()
control_vars = ['ln_Population', 'ln_Land_Area', 'ln_road_km', 'ln_paved_km']

print()

from linearmodels.panel import PanelOLS

results_summary = []
fitted_models   = {}

formula_rhs = 'did_treat + ' + ' + '.join(control_vars)

for log_col, (raw_col, label) in outcome_vars.items():

    reg_cols = [log_col, 'did_treat'] + control_vars
    df_reg   = df_panel[reg_cols].dropna()

    if len(df_reg) == 0:
        continue

    n_treated_obs = df_reg.reset_index()['country'].isin(TREAT_COUNTRIES).sum()
    if n_treated_obs == 0:
        continue

    try:
        model = PanelOLS.from_formula(
            f'{log_col} ~ {formula_rhs} + EntityEffects + TimeEffects',
            data=df_reg,
            drop_absorbed=True
        )
        result = model.fit(cov_type='robust')
        fitted_models[log_col] = result

        coef      = result.params['did_treat']
        se        = result.std_errors['did_treat']
        tstat     = result.tstats['did_treat']
        pval      = result.pvalues['did_treat']
        ci_low    = result.conf_int().loc['did_treat', 'lower']
        ci_up     = result.conf_int().loc['did_treat', 'upper']
        r2_bw     = result.rsquared_between
        nobs      = int(result.nobs)

        stars = ('***' if pval < 0.01 else
                 '**'  if pval < 0.05 else
                 '*'   if pval < 0.10 else '')
        pct_effect = (np.exp(coef) - 1) * 100

        results_summary.append({
            'Dependent Variable':    log_col,
            'Indicator Label':      label,
            'DID Coefficient (β)':  round(coef, 4),
            'Standard Error':       round(se,   4),
            't-Statistic':          round(tstat, 3),
            'p-Value':              round(pval,  4),
            '95% CI Lower Bound':   round(ci_low, 4),
            '95% CI Upper Bound':   round(ci_up,  4),
            'Significance':         stars,
            'Effect Size (%) approx.': round(pct_effect, 2),
            'Observations':         nobs,
        })

        print(f"{'─'*70}")
        print(f"     β(did_treat) = {coef:.4f}{stars}")
        print(f"     95% CI       = [{ci_low:.4f}, {ci_up:.4f}]")

    except Exception as e:
        pass

print(f"\n{'═'*70}")

In [ ]:
# Cell | DID result table and coefficient forest plot

df_results = pd.DataFrame(results_summary)

print("=" * 85)
print("=" * 85)

display_cols = ['Dependent Variable', 'Indicator Label', 'DID Coefficient (β)', 'Standard Error',
                'p-Value', 'Significance', '95% CI Lower Bound', '95% CI Upper Bound', 'Effect Size (%) approx.', 'Observations']
print(df_results[display_cols].to_string(index=False))

for _, row in df_results.iterrows():
    sig_text = {
        '***': 'Significant at 1% level',
        '**':  'Significant at 5% level',
        '*':   'Significant at 10% level',
        '':    'Not statistically significant'
    }.get(row['Significance'], 'Not significant')
    direction = 'Positive' if row['DID Coefficient (β)'] > 0 else 'Negative'

var_labels = {
    'ln_GDP':      'ln(GDP)',
    'ln_GDP_PC':   'ln(GDP per capita)',
    'ln_HFCE_PC':  'ln(HFCE per capita)',
    'ln_Agri_va':  'ln(AVA)',
    'ln_Indus_va': 'ln(IVA)',
    'ln_Serv_va':  'ln(SVA)',
}

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 14
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['figure.dpi'] = 300

fig, ax = plt.subplots(figsize=(10, 5.5))
y_ticks = np.arange(len(df_results))

def get_color(stars):
    return {'***': '#1a5fa8', '**': '#3a79b8', '*': '#6baed6'}.get(stars, '#AAAAAA')

for i, (_, row) in enumerate(df_results.iterrows()):
    coef   = row['DID Coefficient (β)']
    ci_low = row['95% CI Lower Bound']
    ci_up  = row['95% CI Upper Bound']
    stars  = row['Significance']
    color  = get_color(stars)

    ax.plot([ci_low, ci_up], [i, i],
            color=color, lw=2.5, zorder=3, solid_capstyle='round')
    marker = 'D' if stars else 'o'
    ax.scatter([coef], [i],
               color=color, s=90, zorder=5, marker=marker,
               edgecolors='white', linewidths=0.8)
    ax.text(ci_up, i,
            f"  {coef:+.3f}{stars}",
            va='center', ha='left', fontsize=12, color=color, fontweight='bold')

ax.axvline(x=0, color='#CC3333', lw=1.5, ls='--', alpha=0.8,
           label='β = 0 (no effect)', zorder=2)

ax.set_yticks(y_ticks)
ax.set_yticklabels([var_labels.get(r['Dependent Variable'], r['Dependent Variable'])
                    for _, r in df_results.iterrows()], fontsize=14)
ax.set_xlabel(f'Coefficient β (95% CI)',
              fontsize=14)
# ax.set_title(f'Economic Effects of Addis Ababa–Djibouti Railway ({POLICY_YEAR})\n'
#              f'PSM-DID Two-Way Fixed Effects',
#              fontsize=13, fontweight='bold', pad=10)
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.25, ls='--', zorder=1)

legend_elements = [
    plt.scatter([], [], color='#1a5fa8', s=70, marker='D', label='p<0.01 (***)',
                edgecolors='white'),
    plt.scatter([], [], color='#3a79b8', s=70, marker='D', label='p<0.05 (**)',
                edgecolors='white'),
    plt.scatter([], [], color='#6baed6', s=70, marker='D', label='p<0.10 (*)',
                edgecolors='white'),
    plt.Line2D([0], [0], color='#CC3333', lw=1.5, ls='--', label='β = 0'),
]
ax.legend(handles=legend_elements, fontsize=13, loc='upper right',
          framealpha=0.9, edgecolor='#CCCCCC', fancybox=False)
ax.set_xlim(-0.02, 0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Cell | Event-study parallel-trend assessment

print()

# Index observations by years relative to the 2018 railway opening for the event study.
df_did['rel_time'] = df_did['year'] - POLICY_YEAR

rel_times      = sorted(df_did['rel_time'].unique().tolist())
BASE_REL_TIME  = -1
rel_times_excl = [t for t in rel_times if t != BASE_REL_TIME]

for t in rel_times_excl:
    col_name = f'evt_{t:+d}'.replace('+', 'p').replace('-', 'm')
    df_did[col_name] = df_did['treated_dummy'] * (df_did['rel_time'] == t).astype(int)

evt_cols = [f'evt_{t:+d}'.replace('+', 'p').replace('-', 'm') for t in rel_times_excl]

print()

EVENT_STUDY_VARS = ['ln_GDP', 'ln_GDP_PC', 'ln_HFCE_PC']

var_en = {
    'ln_GDP':     'ln(GDP)',
    'ln_GDP_PC':  'ln(GDP per capita)',
    'ln_HFCE_PC': 'ln(HFCE per capita)',
}

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 14
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['figure.dpi'] = 300

fig, axes = plt.subplots(1, len(EVENT_STUDY_VARS), figsize=(15, 5), sharey=False)

for ax_i, es_var in enumerate(EVENT_STUDY_VARS):
    ax = axes[ax_i]

    es_cols   = [es_var, 'did_treat'] + control_vars + evt_cols
    df_es     = df_did[['country', 'year'] + es_cols].dropna(
        subset=[es_var] + control_vars).copy()
    df_es_panel = df_es.set_index(['country', 'year'])

    evt_formula  = ' + '.join(evt_cols)
    ctrl_formula = ' + '.join(control_vars)
    es_formula   = f'{es_var} ~ {evt_formula} + {ctrl_formula} + EntityEffects + TimeEffects'

    try:
        es_result = PanelOLS.from_formula(
            es_formula, data=df_es_panel, drop_absorbed=True
        ).fit(cov_type='robust')

        coefs, ci_lows, ci_ups, tau_vals = [], [], [], []

        for t, col_name in zip(rel_times_excl, evt_cols):
            if col_name in es_result.params.index:
                c    = es_result.params[col_name]
                ci_l = es_result.conf_int().loc[col_name, 'lower']
                ci_u = es_result.conf_int().loc[col_name, 'upper']
            else:
                c, ci_l, ci_u = 0.0, 0.0, 0.0
            coefs.append(c); ci_lows.append(ci_l)
            ci_ups.append(ci_u); tau_vals.append(t)

        insert_idx    = sorted(rel_times).index(BASE_REL_TIME)
        tau_vals_full = tau_vals[:insert_idx] + [BASE_REL_TIME] + tau_vals[insert_idx:]
        coefs_full    = coefs[:insert_idx]    + [0.0]           + coefs[insert_idx:]
        ci_lows_full  = ci_lows[:insert_idx]  + [0.0]           + ci_lows[insert_idx:]
        ci_ups_full   = ci_ups[:insert_idx]    + [0.0]           + ci_ups[insert_idx:]

        ax.fill_between(tau_vals_full, ci_lows_full, ci_ups_full,
                        alpha=0.20, color='#3A79B8', label='95% CI')
        ax.plot(tau_vals_full, coefs_full,
                color='#1a5fa8', lw=2.2, marker='o', ms=7,
                markerfacecolor='white', markeredgewidth=2,
                zorder=5)
        ax.scatter([BASE_REL_TIME], [0], color='#CC3333', s=120,
                   marker='s', zorder=6, label='Ref. year (2017)')
        ax.axhline(y=0,   color='#555', lw=1.0, ls='--', alpha=0.6)
        ax.axvline(x=-0.5, color='#CC3333', lw=1.5, ls=':',
                   alpha=0.7, label='Policy onset')
        ax.axvspan(tau_vals_full[0] - 0.3, -0.5, alpha=0.06,
                   color='#3A9E6F', label='Pre-policy (parallel trend test)')
        ax.axvspan(-0.5, tau_vals_full[-1] + 0.3, alpha=0.06,
                   color='#E07B39', label='Post-policy (treatment effect)')

        # ax.set_title(f'Outcome: {var_en.get(es_var, es_var)}',
        #              fontsize=12, fontweight='bold')
        ax.set_ylabel(f'β_t  ({var_en.get(es_var, es_var)})', fontsize=14)
        ax.set_xticks(tau_vals_full)
        ax.set_xticklabels([f't={t}' for t in tau_vals_full],
                            fontsize=14, rotation=0)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(axis='y', alpha=0.25, ls='--')
        ax.set_ylim(-0.32, 0.52)

        if ax_i == 0:
            ax.legend(fontsize=12, loc='upper left', framealpha=0.9,
                      edgecolor='#CCCCCC', fancybox=False, ncol=1)

        print(f"  {es_var} — pre-policy coefficients (t < 0, expect n.s.):")
        for t, c, cl, cu in zip(tau_vals_full, coefs_full, ci_lows_full, ci_ups_full):
            if t < 0 and t != BASE_REL_TIME:
                flag = "✓ CI includes 0" if cl <= 0 <= cu else "✗ CI excludes 0"
                print(f"    t={t:+d} ({POLICY_YEAR+t}): β={c:+.4f}  95%CI=[{cl:.4f},{cu:.4f}]  {flag}")
        print()

    except Exception as e:
        print(f"  ✗ {es_var} event study failed: {e}")
        ax.set_title(f'{es_var} — failed', fontsize=11)

# fig.suptitle(f'Parallel Trend Test — Event Study\n'
#              f'Addis Ababa–Djibouti Railway ({POLICY_YEAR}): Dynamic Treatment Effects',
#              fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print()


In [ ]:
# Cell | Placebo and exposure-threshold sensitivity analyses

import itertools
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

print()

print()

# Re-estimate the design across alternative railway-growth thresholds.
threshold_results = []

for thresh in [100, 200, 300, 500, 1000]:
    excl_delta  = rail_delta[rail_delta['rail_delta_abs'] > thresh]['country'].tolist()
    pool_thresh = [c for c in candidate_pool if c not in excl_delta]
    thresh_psm_countries = TREAT_COUNTRIES + pool_thresh

    df_pre_th  = df_main[
        df_main['country'].isin(thresh_psm_countries) &
        df_main['year'].isin(PRE_YEARS)
    ].copy()
    df_base_th = df_pre_th.groupby('country')[RAW_EXTRACT_COLS].mean().reset_index()

    for feat in LOG_FEATURES_RAW:
        df_base_th['ln_' + feat] = np.log(df_base_th[feat].clip(lower=1e-6))
    df_base_th['treated'] = df_base_th['country'].isin(TREAT_COUNTRIES).astype(int)

    for feat in PSM_FEATURES:
        if df_base_th[feat].isnull().any():
            for grp in [0, 1]:
                mask_grp = df_base_th['treated'] == grp
                med_val  = df_base_th.loc[mask_grp, feat].median()
                fill_idx = mask_grp & df_base_th[feat].isnull()
                if fill_idx.sum() > 0:
                    fill_val = med_val if pd.notna(med_val) else df_base_th[feat].median()
                    df_base_th.loc[fill_idx, feat] = fill_val
    df_base_th.dropna(subset=PSM_FEATURES, inplace=True)

    if len(df_base_th[df_base_th['treated'] == 1]) == 0:
        print(f"    threshold={thresh}km: no treated obs, skip")
        continue

    X_th  = StandardScaler().fit_transform(df_base_th[PSM_FEATURES].values)
    y_th  = df_base_th['treated'].values
    logit_th = LogisticRegression(
        penalty=None, solver='lbfgs', max_iter=2000, random_state=42
    ).fit(X_th, y_th)
    df_base_th['ps'] = logit_th.predict_proba(X_th)[:, 1]

    df_t_ps = df_base_th[df_base_th['treated'] == 1].copy()
    df_c_ps = df_base_th[df_base_th['treated'] == 0].copy()
    matched_ctrls = []
    
    match_k = 1 if thresh < 100 else N_MATCH
    for _, tr in df_t_ps.iterrows():
        df_c_ps_temp = df_c_ps.copy()
        df_c_ps_temp['dist'] = (df_c_ps_temp['ps'] - tr['ps']).abs()
        matched_ctrls.extend(df_c_ps_temp.nsmallest(match_k, 'dist')['country'].tolist())
    final_thresh_ctrl      = list(set(matched_ctrls))
    final_thresh_countries = TREAT_COUNTRIES + final_thresh_ctrl
    
    # Continue gracefully if this estimation step fails.
    try:
        sb_df = compute_std_bias(df_t_ps, df_base_th[df_base_th['country'].isin(final_thresh_ctrl)], PSM_FEATURES)
        max_sb = sb_df['Standardized Bias (%)'].abs().max()
    except:
        max_sb = 999.9

    df_th = df_main[df_main['country'].isin(final_thresh_countries)].copy()
    df_th['treated_dummy'] = df_th['country'].isin(TREAT_COUNTRIES).astype(int)
    df_th['post_dummy']    = (df_th['year'] >= POLICY_YEAR).astype(int)
    df_th['did_treat']     = df_th['treated_dummy'] * df_th['post_dummy']
    df_th['ln_GDP']        = np.log(df_th['GDP'].clip(lower=1e-6))
    df_th['ln_Population'] = np.log(df_th['Population'].clip(lower=1))
    df_th['ln_Land_Area']  = np.log(df_th['Land_Area_km2'].clip(lower=1))
    df_th['ln_road_km']    = np.log(df_th['total_road_km'].clip(lower=1))
    df_th['ln_paved_km']   = np.log(df_th['motorway_trunk_paved_km'].clip(lower=1))

    df_tp = df_th.set_index(['country', 'year'])[
        ['ln_GDP', 'did_treat'] + control_vars
    ].dropna()

    try:
        r = PanelOLS.from_formula(
            f'ln_GDP ~ did_treat + {" + ".join(control_vars)} + EntityEffects + TimeEffects',
            data=df_tp, drop_absorbed=True
        ).fit(cov_type='robust')

        coef_t  = r.params['did_treat']
        pval_t  = r.pvalues['did_treat']
        stars_t = ('***' if pval_t < 0.01 else
                   '**'  if pval_t < 0.05 else
                   '*'   if pval_t < 0.10 else '')

        threshold_results.append({
            'Rail thresh (km)': thresh,
            'Match k':          match_k,
            'Pool size':        len(pool_thresh),
            'Controls count':   len(final_thresh_ctrl),
            'Max |SB| (%)':     round(max_sb, 1),
            'β(did_treat)':     round(coef_t, 4),
            'p-value':          round(pval_t, 4),
            'Sig.':             stars_t,
        })
        print(f"    threshold={thresh:3d}km | pool={len(pool_thresh):2d} | "
              f"ctrl={len(final_thresh_ctrl):2d} | Max |SB|={max_sb:4.1f}% | "
              f"β={coef_t:+.4f}{stars_t:<3s} | p={pval_t:.4f}")
    except Exception as e:
        print(f"    threshold={thresh}km failed: {e}")

if threshold_results:
    print()
    print(pd.DataFrame(threshold_results).to_string(index=False))

print()
print("=" * 70)
print("=" * 70)
print("""Helper function.""")


In [ ]:
# Cell | Sensitivity-result summary
# Re-estimate the design across alternative railway-growth thresholds.
df_threshold = pd.DataFrame(threshold_results)

numeric_df = df_threshold.select_dtypes(include="number")

summary = pd.DataFrame({
    "mean": numeric_df.mean(),
    "s.d.": numeric_df.std()
}).round(3)

summary